# 실습 04 · DiT와 플로 매칭 — 최신 디노이저의 구조와 학습 목표

장표 「DiT」 · 「DiT 텐서 흐름」 · 「플로 매칭」 세 장(정답 · 스텝 수와 시프트 · ComfyUI 화면)과 Qwen-Image · Z-Image 분석 장을 확인한다. 05 에서 돌릴 세 모델(FLUX.1-dev · Qwen-Image · Z-Image)을 **올리지 않고 파일의 웨이트 이름 · 모양만 읽어** 구조를 확인하고, 플로 매칭은 2차원 장난감으로 직접 학습시켜 본다.

| 셀 | 하는 일 | 장표 |
|---|---|---|
| 1 | 설정 | – |
| 2 | 준비 | – |
| 3 | latent 를 패치로 자르기 — [16, 128, 128] → 토큰 4,096개 × 64, 되접기 | DiT 2번 · DiT 텐서 흐름 |
| 4 | 웨이트 모양만 읽어 구조 확인 — 패치 투영 · 글 투영 · 블록 · 출력층 | DiT 4~6번 · Qwen-Image 분석 · 세 모델 비교 |
| 5 | 어텐션 비용표 — 해상도별 토큰 수와 확률표 칸 수 | DiT 3번 |
| 6 | 플로 매칭 장난감 — 노이즈 예측 vs 직선 속도 예측, 스텝 수별 품질 · 경로 | 플로 매칭 — 정답 ε − z_0 · 스텝 수와 시프트 |
| 7 | 시프트 — 노이즈 수준 간격을 어디에 몰아 줄까 | 플로 매칭 — 스텝 수와 시프트 · 플로 매칭 — ComfyUI 화면(「모델 샘플링(AuraFlow)」) |

- 실행 : **Run → Run All Cells**. GPU 필수(셀 6 학습). 전체 약 1~2분. 모델 파일은 읽기만 한다(메모리에 올리지 않음)

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
import os
MODELS_DIR = os.environ.get("MODELS_DIR", "/models")
MODELS = {"FLUX.1-dev":  f"{MODELS_DIR}/diffusion_models/flux1-dev-Q4_K_S.gguf",
          "Qwen-Image":  f"{MODELS_DIR}/diffusion_models/qwen-image-Q4_K_M.gguf",
          "Z-Image":     f"{MODELS_DIR}/diffusion_models/z_image_bf16.safetensors"}
SIZES = [512, 1024, 1328]          # 셀 5 : 그림 한 변(픽셀)
TEXT_TOKENS = 100                  # 셀 5 : 프롬프트 토큰 수 가정
TOY_STEPS = [1, 2, 4, 8, 16, 64]   # 셀 6 : 생성 스텝 수
SHIFTS = [1.0, 3.0, 3.1, 6.0]      # 셀 7 : 05 의 Qwen-Image 3.1 · Z-Image 3.0

### 셀 2 · 준비

In [ ]:
# 셀 2 · 준비
import math, re, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, gguf
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import font_manager
from safetensors import safe_open

have = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams.update({"font.family": [f for f in ("NanumGothic", "NanumBarunGothic", "Malgun Gothic") if f in have] or ["DejaVu Sans"],
                     "axes.unicode_minus": False})
assert torch.cuda.is_available(), "GPU 가 보이지 않음 — 5-2 는 NVIDIA GPU 필수(README 「실행 환경」)"
DEV = "cuda"


def weight_shapes(path):
    """모델 파일에서 {웨이트 이름: 모양} 만 읽음 — 값은 읽지 않는다"""
    if path.endswith(".gguf"):
        return {t.name: [int(x) for x in reversed(t.shape)] for t in gguf.GGUFReader(path).tensors}
    with safe_open(path, "pt") as f:
        return {k: list(f.get_slice(k).get_shape()) for k in f.keys()}

### 셀 3 · latent 를 패치로 자르기
- 1024×1024 그림 → 16채널 VAE(03 셀 8) → latent [16, 128, 128]
- DiT 는 이것을 **2×2 칸 패치**로 자른다 → (128/2) × (128/2) = **4,096개 토큰**, 토큰 하나 = 2×2×16 = **64개 값**
- 5-1 의 ViT : 224×224 **사진**을 32×32 픽셀 패치로 → 49개 × 3,072값. DiT 는 같은 일을 **latent** 에 한다
- 디노이저의 출력도 토큰마다 64개 값 → **되접으면** 원래 latent 모양 = 지울 노이즈(또는 속도). 풀링 없음 — 5-1 은 토큰들을 벡터 하나로 줄였지만 여기서는 토큰 하나하나가 그 자리의 답

In [ ]:
# 셀 3 · latent 를 패치로 자르기
def patchify(z, p=2):
    c, h, w = z.shape
    return z.reshape(c, h // p, p, w // p, p).permute(1, 3, 2, 4, 0).reshape((h // p) * (w // p), p * p * c)

def unpatchify(t, c, h, w, p=2):
    return t.reshape(h // p, w // p, p, p, c).permute(4, 0, 2, 1, 3).reshape(c, h, w)

rows = []
for name, (c, h, w) in {"SD1.5 latent (512²)": (4, 64, 64), "FLUX · Qwen · Z latent (1024²)": (16, 128, 128)}.items():
    z = torch.randn(c, h, w)
    tks = patchify(z)
    back = unpatchify(tks, c, h, w)
    rows.append((name, f"[{c}, {h}, {w}]", f"{tks.shape[0]:,} × {tks.shape[1]}", f"{(back - z).abs().max().item():.0f}"))
rows.append(("5-1 ViT-B/32 사진 (224²)", "[3, 224, 224]", "49 × 3,072", "–"))
print(pd.DataFrame(rows, columns=["입력", "모양", "토큰 수 × 토큰 길이", "되접기 차이"]).to_string(index=False))

### 셀 4 · 웨이트 모양만 읽어 구조 확인
- 모델 파일 머리에는 웨이트마다 이름 · 모양이 적혀 있다. 값을 올리지 않고 이것만 읽어도 구조가 보인다
- 보는 곳
  - **패치 투영** : 입력 폭 64(= 셀 3 의 토큰 길이) → 모델 폭
  - **글 투영** : 입력 폭 = **조건 인코더의 출력 폭** → 인코더를 바꾸면 이 행렬이 맞지 않는다(02 셀 10 과 같은 이야기)
  - **블록** : 이중 스트림(글 · 그림이 웨이트를 따로 쓰고 어텐션만 함께 — MMDiT) / 단일 스트림(한 웨이트로 한 줄)
  - **출력층** : 모델 폭 → 64 = 토큰마다 2×2×16 값을 돌려줌
  - **timestep 입력** 256 : 01 셀 9 와 같은 사인 · 코사인 벡터

In [ ]:
# 셀 4 · 웨이트 모양만 읽어 구조 확인
def first(S, *pats):
    for p in pats:
        for k, v in S.items():
            if re.fullmatch(p, k): return k, v
    return None, None

def count_blocks(S, prefix):
    return len({m.group(1) for k in S for m in [re.match(rf"{prefix}\.(\d+)\.", k)] if m})

rows = []
for name, path in MODELS.items():
    S = weight_shapes(path)
    n = sum(math.prod(v) for v in S.values())
    _, pin = first(S, r"img_in\.weight", r"x_embedder\.weight")
    _, tin = first(S, r"txt_in\.weight", r"cap_embedder\.1\.weight")
    _, vin = first(S, r"vector_in\.in_layer\.weight")
    _, out = first(S, r"final_layer\.linear\.weight", r"proj_out\.weight")
    _, tim = first(S, r"time_in\.in_layer\.weight", r"time_text_embed\.timestep_embedder\.linear_1\.weight", r"t_embedder\.mlp\.0\.weight")
    blocks = {"이중 스트림": count_blocks(S, "double_blocks") or count_blocks(S, "transformer_blocks"),
              "단일 스트림": count_blocks(S, "single_blocks") or count_blocks(S, "layers")}
    rows.append((name, f"{n / 1e9:.2f}B", f"{pin[1]} → {pin[0]}", f"{tin[1]} → {tin[0]}",
                 f"{vin[1]} → {vin[0]}" if vin else "–", f"{blocks['이중 스트림']} + {blocks['단일 스트림']}",
                 f"{out[1]} → {out[0]}", tim[1]))
print(pd.DataFrame(rows, columns=["모델", "파라미터", "패치 투영", "글 투영", "풀링 벡터 투영", "블록 이중+단일",
                                  "출력층", "timestep 입력"]).to_string(index=False))
print("\n글 투영 입력 폭 : 4096 = T5-XXL · 3584 = Qwen2.5-VL-7B · 2560 = Qwen3-4B 의 출력 폭  /  풀링 벡터 768 = CLIP-L 의 [EOS] 벡터")

### 셀 5 · 어텐션 비용표
- DiT 는 모든 토큰이 서로를 본다 → 어텐션 확률표 = (그림 토큰 + 글 토큰)² 칸 — **해상도가 2배면 토큰 4배, 칸 16배**
- SD1.5 U-Net 도 가장 큰 해상도(64×64)에서 셀프 어텐션을 한다(02 셀 5) — 그림 칸 수² 는 같은 식으로 늘어난다
- 장표 「DiT」 3번 단점 "토큰 수가 많아지면 attention 비용이 커짐"을 숫자로
- 장표의 1,680만 = 1024² 그림 토큰 4,096개만의 4,096². 아래 표는 글 토큰 100개를 더한 (4,096 + 100)² = 17.6M

In [ ]:
# 셀 5 · 어텐션 비용표
rows = []
for s in SIZES:
    lat = s // 8
    img_tok = (lat // 2) ** 2
    tot = img_tok + TEXT_TOKENS
    unet_tok = lat ** 2                        # SD1.5 U-Net 최고 해상도 셀프 어텐션(패치 없이 latent 칸마다)
    rows.append((f"{s}×{s}", f"{lat}×{lat}", f"{img_tok:,}", f"{tot:,}", f"{tot ** 2 / 1e6:,.1f}M", f"{unet_tok:,}", f"{unet_tok ** 2 / 1e6:,.1f}M"))
print(pd.DataFrame(rows, columns=["그림", "latent", "DiT 그림 토큰", f"+ 글 {TEXT_TOKENS}", "DiT 확률표 칸/헤드·층",
                                  "U-Net 64×64급 칸", "U-Net 셀프 어텐션 칸"]).to_string(index=False))

### 셀 6 · 플로 매칭 장난감 — 노이즈 예측 vs 직선 속도 예측
- 2차원 점 구름(원 위의 점 무리 8개)을 만드는 생성기를 **같은 크기의 작은 신경망** 두 개로 학습한다
  - **노이즈 예측**(01 의 방식) : `z_t = a_t·x₀ + b_t·ε`, 정답 = ε. 생성은 DDIM 식 이동
  - **플로 매칭(직선 속도 예측)** : `z_t = (1−t)·x₀ + t·ε` — 데이터와 노이즈를 **직선**으로 잇는다. 정답 = 그 직선의 방향 `v = ε − x₀`. 생성은 t = 1 → 0 으로 v 를 따라 한 걸음씩(오일러법)
- 잰다 : ① 스텝 수별로 만든 점들이 데이터와 얼마나 가까운가(최근접 거리 평균, 작을수록 좋음) ② 생성 경로의 곧음(시작~끝 거리 / 경로 길이, 1 = 직선)
- 장표 「플로 매칭 — 정답 ε − z_0」 · 「스텝 수와 시프트」 : 학습 때 잇는 선이 직선이라 큰 걸음으로 가도 덜 빗나감 → 적은 스텝. SD3 · FLUX · Qwen-Image · Z-Image 가 모두 이 방식
- 주의 : 곧은 것은 **학습 때 잇는 선**이다. 생성 경로는 여러 직선을 평균한 방향을 따라가 휠 수 있다(②) — 더 곧게 펴려면 한 번 더 학습하는 단계(reflow · 증류)가 따로 있다

In [ ]:
# 셀 6 · 플로 매칭 장난감
torch.manual_seed(0)
def data(n):                                   # 반지름 2 원 위의 점 무리 8개
    k = torch.randint(0, 8, (n,)); ang = k * 2 * math.pi / 8
    return torch.stack([2 * torch.cos(ang), 2 * torch.sin(ang)], 1) + 0.1 * torch.randn(n, 2)

class Net(nn.Module):
    def __init__(s, h=256):
        super().__init__()
        s.f = nn.Sequential(nn.Linear(2 + 32, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(s, x, t):                      # t ∈ [0, 1] → 사인 · 코사인 16쌍(01 셀 9 와 같은 발상)
        f = torch.exp(torch.linspace(0, 4, 16, device=x.device))
        return s.f(torch.cat([x, torch.sin(t[:, None] * f), torch.cos(t[:, None] * f)], 1))

abar = torch.cumprod(1 - torch.linspace(1e-4, 0.02, 1000), 0).to(DEV)    # 노이즈 예측 쪽 스케줄(DDPM 1000단계)
def train(kind, iters=6000):
    net = Net().to(DEV); opt = torch.optim.Adam(net.parameters(), 1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iters)
    for _ in range(iters):
        x0 = data(2048).to(DEV); e = torch.randn_like(x0)
        if kind == "eps":
            i = torch.randint(0, 1000, (2048,), device=DEV); a, b = abar[i].sqrt()[:, None], (1 - abar[i]).sqrt()[:, None]
            loss = ((net(a * x0 + b * e, i / 999) - e) ** 2).mean()
        else:
            t = torch.rand(2048, device=DEV)[:, None]
            loss = ((net((1 - t) * x0 + t * e, t[:, 0]) - (e - x0)) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    return net

@torch.no_grad()
def sample(net, kind, n_steps, z):
    path = [z]
    if kind == "eps":
        ts = torch.linspace(999, 0, n_steps + 1, device=DEV).round().long()
        for i in range(n_steps):
            t, tn = ts[i], ts[i + 1]
            e = net(z, torch.full((len(z),), t / 999, device=DEV))
            x0 = (z - (1 - abar[t]).sqrt() * e) / abar[t].sqrt()
            z = abar[tn].sqrt() * x0 + (1 - abar[tn]).sqrt() * e
            path.append(z)
    else:
        ts = torch.linspace(1, 0, n_steps + 1, device=DEV)
        for i in range(n_steps):
            z = z + (ts[i + 1] - ts[i]) * net(z, torch.full((len(z),), float(ts[i]), device=DEV))
            path.append(z)
    return z, torch.stack(path)

def dist(a, b):                                # 생성 점마다 가장 가까운 데이터 점까지 + 반대 방향, 평균
    d = torch.cdist(a, b)
    return (d.min(1).values.mean() + d.min(0).values.mean()).item() / 2

t0 = time.time(); nets = {k: train(k) for k in ("eps", "flow")}; t_train = time.time() - t0
ref = data(3000).to(DEV); z0 = torch.randn(3000, 2, generator=torch.Generator().manual_seed(1)).to(DEV)
rows, keep = [], {}
for n in TOY_STEPS:
    r = [n]
    for k in ("eps", "flow"):
        x, p = sample(nets[k], k, n, z0)
        seg = (p[1:] - p[:-1]).norm(dim=-1).sum(0)
        r += [f"{dist(x, ref):.3f}", f"{((p[-1] - p[0]).norm(dim=-1) / seg).mean():.2f}"]
        keep[(k, n)] = x.cpu()
    rows.append(r)
print(f"학습 : 두 모델 각 6,000걸음 · {t_train:.0f}초 (같은 신경망 크기)")
print(pd.DataFrame(rows, columns=["스텝", "노이즈 예측 거리", "곧음", "플로 매칭 거리", "곧음"]).to_string(index=False))
show_n = [2, 4, 8, 64]
fig, ax = plt.subplots(2, len(show_n) + 1, figsize=(3 * (len(show_n) + 1), 6))
for r, (k, lab) in enumerate((("eps", "노이즈 예측"), ("flow", "플로 매칭"))):
    ax[r, 0].scatter(*ref.cpu().T, s=1, c="k"); ax[r, 0].set_title("데이터", fontsize=9)
    for a, n in zip(ax[r, 1:], show_n):
        a.scatter(*keep[(k, n)].T, s=1); a.set_title(f"{lab} · {n}스텝", fontsize=9)
for a in ax.ravel(): a.set_xlim(-3, 3); a.set_ylim(-3, 3); a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

### 셀 7 · 시프트 — 노이즈 수준 간격을 어디에 몰아 줄까
- 플로 매칭 모델은 노이즈 수준 σ(장표의 t 와 같은 값. ComfyUI 코드에서는 σ)를 1 → 0 으로 내려가며 스텝을 밟는다. 시프트는 그 간격을 바꾼다 : `σ' = s·σ / (1 + (s − 1)·σ)` (ComfyUI `comfy/model_sampling.py` 의 `time_snr_shift`)
- s > 1 이면 σ 가 큰(노이즈 많은) 구간에 스텝이 더 몰린다 → 구도를 정하는 초반(01 셀 6 · 02 셀 8)에 더 많은 걸음. 해상도가 클수록 같은 σ 에서도 그림 정보가 더 남아 있어 큰 시프트를 쓴다(SD3 논문)
- 05 의 워크플로 : Qwen-Image 3.1 · Z-Image 3.0 (ComfyUI 노드 「모델 샘플링(AuraFlow)」)

In [ ]:
# 셀 7 · 시프트
N = 20
sig = torch.linspace(1, 0, N + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
rows = []
for s in SHIFTS:
    sp = s * sig / (1 + (s - 1) * sig)
    ax[0].plot(range(N + 1), sp, "o-", ms=3, label=f"shift {s}")
    rows.append((s, int((sp[:-1] > 0.5).sum()), int((sp[:-1] > 0.8).sum())))
ax[0].set_xlabel("스텝"); ax[0].set_ylabel("노이즈 수준 σ"); ax[0].legend(fontsize=8)
x = torch.linspace(0, 1, 101)
for s in SHIFTS: ax[1].plot(x, s * x / (1 + (s - 1) * x), label=f"shift {s}")
ax[1].set_xlabel("원래 σ"); ax[1].set_ylabel("바뀐 σ"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows, columns=["shift", f"{N}스텝 중 σ>0.5", "σ>0.8"]).to_string(index=False))